# Lab 3: Does the playbook help? Measure it.

**Cost: about $0.20 to $0.60 (60 answers by Haiku 5.5 plus 60 grades by Sonnet 5.5). Time: 15 minutes.**

Same 20 questions a close team would ask at the start of the November close, answered three ways by the **same model**:

| Condition | Memory available | Layer of the hierarchy |
|---|---|---|
| `none` | nothing | stateless baseline |
| `transcripts` | read-only search over all 17 raw transcripts | transcripts |
| `playbook` | read-only search over the dreamed playbook | representation |

Each answer is checked for required details and graded by a judge against the expected **current** answer
(`data/eval_questions.yaml`). Question types: stale fact, buried detail, recurring trap, process, synthesis, integrity.

This mirrors the article's experiments (memory vs. stateless baseline; wiki vs. search vs. both) at toy scale. Treat
the numbers as illustrative: 20 questions, one run.

In [ ]:
from close_dreamer import config
from close_dreamer import evaluate as ev

PLAYBOOK = config.OUTPUT / "lab-playbook"   # from Lab 2; use config.PLAYBOOK for the CLI's playbook
questions = ev.load_questions(config.DATA / "eval_questions.yaml")
print(len(questions), "questions;", sorted({q['type'] for q in questions}))

In [ ]:
# Paid cell.
client = config.make_client()
graded, tracker = ev.run(client, questions, transcripts_dir=config.TRANSCRIPTS, playbook_dir=PLAYBOOK,
                         budget_usd=2.00, concurrency=4)
print(tracker.table())
print("saved:", ev.save(graded, tracker, config.RUNS))

## Results table
This table is also the accessible view of the chart below.

In [ ]:
from IPython.display import Markdown
Markdown(ev.summarize(graded, tracker))

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# Categorical palette (validated: lightness, chroma, colorblind separation). Fixed order, one color per condition.
COLORS = {"none": "#2a78d6", "transcripts": "#eb6834", "playbook": "#1baf7a"}
LABELS = {"none": "No memory", "transcripts": "Raw transcripts", "playbook": "Dreamed playbook"}
SURFACE, INK, MUTED = "#fcfcfb", "#1f1f1e", "#6b6a63"

scores = defaultdict(list)
for g in graded:
    scores[(g.type, g.condition)].append(g.judge_score)
types = sorted({g.type for g in graded})
conds = [c for c in COLORS if any(g.condition == c for g in graded)]

fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
width = 0.26
for i, c in enumerate(conds):
    xs = [t + (i - (len(conds) - 1) / 2) * width for t in range(len(types))]
    ys = [100 * sum(scores[(t, c)]) / len(scores[(t, c)]) for t in types]
    ax.bar(xs, ys, width=width, color=COLORS[c], edgecolor=SURFACE, linewidth=2, label=LABELS[c], zorder=2)
ax.set_xticks(range(len(types)), [t.replace("_", " ") for t in types], color=INK)
ax.set_ylim(0, 100)
ax.set_ylabel("Judge score (%)", color=MUTED)
ax.yaxis.grid(True, color="#e6e5e0", linewidth=1, zorder=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#cfcdc6")
ax.tick_params(colors=MUTED, length=0)
ax.legend(frameon=False, ncols=3, loc="upper left", bbox_to_anchor=(0, 1.12), labelcolor=INK)
ax.set_title("Answer quality by question type", loc="left", color=INK, pad=28, fontsize=12)
fig.tight_layout()
fig.savefig(config.RUNS / "eval-by-type.png", dpi=200, facecolor=SURFACE)
plt.show()

## Read the answers, not just the scores

In [ ]:
by_id = defaultdict(dict)
for g in graded:
    by_id[g.id][g.condition] = g
for qid in ["q01", "q09", "q19"]:
    q = next(q for q in questions if q["id"] == qid)
    print(f"=== {qid} ({q['type']}): {q['question']}\nexpected: {q['expected']}")
    for c in conds:
        g = by_id[qid][c]
        print(f"--- {c} (score {g.judge_score}): {g.answer[:400]}")
    print()

## What to look for

- **No memory** should fail almost everything specific; it can only guess.
- **Raw transcripts** often find the answer but may return a **superseded** value (BD3 instead of BD2, M instead of
  ZCLS), and cost more tokens because the agent reads long files.
- **Playbook** should do best on stale facts and recurring traps, with fewer tokens. If it doesn't, find out why: open
  the page the agent read. Memory is only as good as the dream that wrote it.

The article found wikis help most on multi-document synthesis and add little when the answer is one searchable line.
Check whether your buried-detail scores show the same thing.

**Write down** the headline numbers for the README results table and the LinkedIn post (`outreach/linkedin-post.md`).